# Options from Zero · Episode 9: From tree to bell curve: what volatility is

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> **Data.** `data/fed_h10_audusd.csv`: AUD/USD (US dollars per Australian dollar), noon buying rates in New York certified by the Federal Reserve Bank of New York, from the Federal Reserve Board's H.10 release. Source: Board of Governors of the Federal Reserve System; Board website content is in the public domain.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. A tree turns into a bell curve · 4. Volatility, defined · 5. A lopsided range · 6. Real data: AUD/USD · 7. Volatility moves · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `data/fed_h10_audusd.csv` (run me first) { display-mode: "form" }
# Source: Board of Governors of the Federal Reserve System, H.10 release (AUD/USD, noon buying rates in
# New York, certified by the Federal Reserve Bank of New York). Board website content is in the public domain.
from pathlib import Path
Path('data/fed_h10_audusd.csv').parent.mkdir(parents=True, exist_ok=True)
Path('data/fed_h10_audusd.csv').write_text("""date,audusd
2018-01-02,0.7824
2018-01-03,0.7840
2018-01-04,0.7856
2018-01-05,0.7859
2018-01-08,0.7838
2018-01-09,0.7822
2018-01-10,0.7841
2018-01-11,0.7885
2018-01-12,0.7886
2018-01-16,0.7958
2018-01-17,0.7982
2018-01-18,0.7993
2018-01-19,0.8001
2018-01-22,0.8004
2018-01-23,0.7984
2018-01-24,0.8066
2018-01-25,0.8089
2018-01-26,0.8105
2018-01-29,0.8089
2018-01-30,0.8076
2018-01-31,0.8069
2018-02-01,0.8028
2018-02-02,0.7929
2018-02-05,0.7926
2018-02-06,0.7877
2018-02-07,0.7842
2018-02-08,0.7789
2018-02-09,0.7809
2018-02-12,0.7836
2018-02-13,0.7857
2018-02-14,0.7873
2018-02-15,0.7912
2018-02-16,0.7925
2018-02-20,0.7903
2018-02-21,0.7833
2018-02-22,0.7850
2018-02-23,0.7827
2018-02-26,0.7843
2018-02-27,0.7804
2018-02-28,0.7801
2018-03-01,0.7762
2018-03-02,0.7754
2018-03-05,0.7759
2018-03-06,0.7822
2018-03-07,0.7796
2018-03-08,0.7787
2018-03-09,0.7849
2018-03-12,0.7864
2018-03-13,0.7881
2018-03-14,0.7877
2018-03-15,0.7816
2018-03-16,0.7733
2018-03-19,0.7705
2018-03-20,0.7691
2018-03-21,0.7708
2018-03-22,0.7696
2018-03-23,0.7721
2018-03-26,0.7727
2018-03-27,0.7717
2018-03-28,0.7681
2018-03-29,0.7672
2018-03-30,0.7690
2018-04-02,0.7658
2018-04-03,0.7674
2018-04-04,0.7701
2018-04-05,0.7688
2018-04-06,0.7684
2018-04-09,0.7708
2018-04-10,0.7762
2018-04-11,0.7765
2018-04-12,0.7757
2018-04-13,0.7768
2018-04-16,0.7777
2018-04-17,0.7768
2018-04-18,0.7784
2018-04-19,0.7751
2018-04-20,0.7671
2018-04-23,0.7614
2018-04-24,0.7598
2018-04-25,0.7559
2018-04-26,0.7560
2018-04-27,0.7566
2018-04-30,0.7543
2018-05-01,0.7488
2018-05-02,0.7501
2018-05-03,0.7513
2018-05-04,0.7534
2018-05-07,0.7526
2018-05-08,0.7445
2018-05-09,0.7456
2018-05-10,0.7515
2018-05-11,0.7544
2018-05-14,0.7554
2018-05-15,0.7477
2018-05-16,0.7504
2018-05-17,0.7508
2018-05-18,0.7511
2018-05-21,0.7565
2018-05-22,0.7595
2018-05-23,0.7534
2018-05-24,0.7567
2018-05-25,0.7558
2018-05-29,0.7514
2018-05-30,0.7574
2018-05-31,0.7570
2018-06-01,0.7566
2018-06-04,0.7648
2018-06-05,0.7606
2018-06-06,0.7677
2018-06-07,0.7630
2018-06-08,0.7601
2018-06-11,0.7616
2018-06-12,0.7607
2018-06-13,0.7604
2018-06-14,0.7514
2018-06-15,0.7455
2018-06-18,0.7419
2018-06-19,0.7386
2018-06-20,0.7387
2018-06-21,0.7374
2018-06-22,0.7424
2018-06-25,0.7406
2018-06-26,0.7402
2018-06-27,0.7373
2018-06-28,0.7355
2018-06-29,0.7399
2018-07-02,0.7322
2018-07-03,0.7385
2018-07-05,0.7388
2018-07-06,0.7421
2018-07-09,0.7466
2018-07-10,0.7464
2018-07-11,0.7400
2018-07-12,0.7418
2018-07-13,0.7410
2018-07-16,0.7416
2018-07-17,0.7392
2018-07-18,0.7392
2018-07-19,0.7332
2018-07-20,0.7410
2018-07-23,0.7383
2018-07-24,0.7416
2018-07-25,0.7412
2018-07-26,0.7384
2018-07-27,0.7410
2018-07-30,0.7408
2018-07-31,0.7438
2018-08-01,0.7402
2018-08-02,0.7379
2018-08-03,0.7406
2018-08-06,0.7393
2018-08-07,0.7428
2018-08-08,0.7420
2018-08-09,0.7394
2018-08-10,0.7298
2018-08-13,0.7270
2018-08-14,0.7240
2018-08-15,0.7233
2018-08-16,0.7276
2018-08-17,0.7282
2018-08-20,0.7308
2018-08-21,0.7352
2018-08-22,0.7352
2018-08-23,0.7264
2018-08-24,0.7332
2018-08-27,0.7346
2018-08-28,0.7352
2018-08-29,0.7292
2018-08-30,0.7262
2018-08-31,0.7192
2018-09-04,0.7174
2018-09-05,0.7189
2018-09-06,0.7198
2018-09-07,0.7135
2018-09-10,0.7118
2018-09-11,0.7107
2018-09-12,0.7161
2018-09-13,0.7192
2018-09-14,0.7178
2018-09-17,0.7186
2018-09-18,0.7220
2018-09-19,0.7273
2018-09-20,0.7276
2018-09-21,0.7278
2018-09-24,0.7263
2018-09-25,0.7251
2018-09-26,0.7260
2018-09-27,0.7218
2018-09-28,0.7238
2018-10-01,0.7223
2018-10-02,0.7199
2018-10-03,0.7138
2018-10-04,0.7078
2018-10-05,0.7048
2018-10-09,0.7092
2018-10-10,0.7100
2018-10-11,0.7112
2018-10-12,0.7116
2018-10-15,0.7142
2018-10-16,0.7140
2018-10-17,0.7130
2018-10-18,0.7118
2018-10-19,0.7132
2018-10-22,0.7086
2018-10-23,0.7074
2018-10-24,0.7072
2018-10-25,0.7084
2018-10-26,0.7088
2018-10-29,0.7074
2018-10-30,0.7112
2018-10-31,0.7090
2018-11-01,0.7189
2018-11-02,0.7199
2018-11-05,0.7198
2018-11-06,0.7221
2018-11-07,0.7282
2018-11-08,0.7286
2018-11-09,0.7226
2018-11-13,0.7220
2018-11-14,0.7219
2018-11-15,0.7269
2018-11-16,0.7312
2018-11-19,0.7291
2018-11-20,0.7246
2018-11-21,0.7267
2018-11-23,0.7225
2018-11-26,0.7230
2018-11-27,0.7206
2018-11-28,0.7233
2018-11-29,0.7314
2018-11-30,0.7311
2018-12-03,0.7360
2018-12-04,0.7353
2018-12-06,0.7216
2018-12-07,0.7210
2018-12-10,0.7190
2018-12-11,0.7204
2018-12-12,0.7223
2018-12-13,0.7228
2018-12-14,0.7176
2018-12-17,0.7182
2018-12-18,0.7172
2018-12-19,0.7172
2018-12-20,0.7104
2018-12-21,0.7072
2018-12-26,0.7070
2018-12-27,0.7020
2018-12-28,0.7044
2018-12-31,0.7046
2019-01-02,0.7000
2019-01-03,0.7014
2019-01-04,0.7108
2019-01-07,0.7143
2019-01-08,0.7134
2019-01-09,0.7172
2019-01-10,0.7190
2019-01-11,0.7204
2019-01-15,0.7192
2019-01-16,0.7184
2019-01-17,0.7171
2019-01-18,0.7172
2019-01-22,0.7133
2019-01-23,0.7133
2019-01-24,0.7096
2019-01-25,0.7172
2019-01-28,0.7166
2019-01-29,0.7149
2019-01-30,0.7196
2019-01-31,0.7282
2019-02-01,0.7260
2019-02-04,0.7218
2019-02-05,0.7234
2019-02-06,0.7128
2019-02-07,0.7093
2019-02-08,0.7090
2019-02-11,0.7065
2019-02-12,0.7094
2019-02-13,0.7113
2019-02-14,0.7091
2019-02-15,0.7122
2019-02-19,0.7146
2019-02-21,0.7080
2019-02-22,0.7138
2019-02-25,0.7169
2019-02-26,0.7182
2019-02-27,0.7134
2019-02-28,0.7099
2019-03-01,0.7080
2019-03-04,0.7084
2019-03-05,0.7073
2019-03-06,0.7023
2019-03-07,0.7032
2019-03-08,0.7048
2019-03-11,0.7056
2019-03-12,0.7082
2019-03-13,0.7087
2019-03-14,0.7061
2019-03-15,0.7089
2019-03-18,0.7100
2019-03-19,0.7102
2019-03-20,0.7088
2019-03-21,0.7110
2019-03-22,0.7091
2019-03-25,0.7114
2019-03-26,0.7140
2019-03-27,0.7070
2019-03-28,0.7079
2019-03-29,0.7104
2019-04-01,0.7117
2019-04-02,0.7058
2019-04-03,0.7124
2019-04-04,0.7112
2019-04-05,0.7099
2019-04-08,0.7124
2019-04-09,0.7129
2019-04-10,0.7158
2019-04-11,0.7128
2019-04-12,0.7176
2019-04-15,0.7178
2019-04-16,0.7174
2019-04-17,0.7168
2019-04-18,0.7150
2019-04-19,0.7150
2019-04-22,0.7134
2019-04-23,0.7096
2019-04-24,0.7026
2019-04-25,0.7010
2019-04-26,0.7044
2019-04-29,0.7054
2019-04-30,0.7040
2019-05-01,0.7038
2019-05-02,0.7002
2019-05-03,0.7018
2019-05-06,0.6995
2019-05-07,0.7004
2019-05-08,0.7004
2019-05-09,0.6976
2019-05-10,0.6998
2019-05-13,0.6948
2019-05-14,0.6946
2019-05-15,0.6930
2019-05-16,0.6905
2019-05-17,0.6874
2019-05-20,0.6913
2019-05-21,0.6876
2019-05-22,0.6880
2019-05-23,0.6896
2019-05-24,0.6914
2019-05-28,0.6930
2019-05-29,0.6912
2019-05-30,0.6910
2019-05-31,0.6930
2019-06-03,0.6955
2019-06-04,0.6987
2019-06-05,0.6974
2019-06-06,0.6984
2019-06-07,0.7002
2019-06-10,0.6964
2019-06-11,0.6956
2019-06-12,0.6942
2019-06-13,0.6910
2019-06-14,0.6872
2019-06-17,0.6860
2019-06-18,0.6872
2019-06-19,0.6870
2019-06-20,0.6917
2019-06-21,0.6920
2019-06-24,0.6964
2019-06-25,0.6975
2019-06-26,0.6990
2019-06-27,0.6998
2019-06-28,0.7009
2019-07-01,0.6963
2019-07-02,0.6992
2019-07-03,0.7033
2019-07-05,0.6968
2019-07-08,0.6968
2019-07-09,0.6927
2019-07-10,0.6964
2019-07-11,0.6982
2019-07-12,0.7007
2019-07-15,0.7036
2019-07-16,0.7020
2019-07-17,0.7021
2019-07-18,0.7043
2019-07-19,0.7042
2019-07-22,0.7042
2019-07-23,0.6999
2019-07-24,0.6978
2019-07-25,0.6954
2019-07-26,0.6908
2019-07-29,0.6908
2019-07-30,0.6872
2019-07-31,0.6888
2019-08-01,0.6858
2019-08-02,0.6806
2019-08-05,0.6780
2019-08-06,0.6770
2019-08-07,0.6754
2019-08-08,0.6812
2019-08-09,0.6796
2019-08-12,0.6756
2019-08-13,0.6798
2019-08-14,0.6742
2019-08-15,0.6784
2019-08-16,0.6782
2019-08-19,0.6772
2019-08-20,0.6777
2019-08-21,0.6792
2019-08-22,0.6764
2019-08-23,0.6749
2019-08-26,0.6782
2019-08-27,0.6758
2019-08-28,0.6746
2019-08-29,0.6734
2019-08-30,0.6730
2019-09-03,0.6758
2019-09-04,0.6796
2019-09-05,0.6820
2019-09-06,0.6856
2019-09-09,0.6874
2019-09-10,0.6868
2019-09-11,0.6858
2019-09-12,0.6871
2019-09-13,0.6886
2019-09-16,0.6865
2019-09-17,0.6858
2019-09-18,0.6844
2019-09-19,0.6795
2019-09-20,0.6770
2019-09-23,0.6772
2019-09-24,0.6793
2019-09-25,0.6750
2019-09-26,0.6754
2019-09-27,0.6764
2019-09-30,0.6746
2019-10-01,0.6703
2019-10-02,0.6698
2019-10-03,0.6751
2019-10-04,0.6764
2019-10-07,0.6742
2019-10-08,0.6727
2019-10-09,0.6730
2019-10-10,0.6770
2019-10-11,0.6804
2019-10-15,0.6757
2019-10-16,0.6755
2019-10-17,0.6826
2019-10-18,0.6846
2019-10-21,0.6867
2019-10-22,0.6863
2019-10-23,0.6847
2019-10-24,0.6817
2019-10-25,0.6829
2019-10-28,0.6839
2019-10-29,0.6862
2019-10-30,0.6872
2019-10-31,0.6888
2019-11-01,0.6914
2019-11-04,0.6898
2019-11-05,0.6892
2019-11-06,0.6881
2019-11-07,0.6895
2019-11-08,0.6854
2019-11-12,0.6846
2019-11-13,0.6831
2019-11-14,0.6780
2019-11-15,0.6810
2019-11-18,0.6820
2019-11-19,0.6826
2019-11-20,0.6797
2019-11-21,0.6793
2019-11-22,0.6784
2019-11-25,0.6770
2019-11-26,0.6787
2019-11-27,0.6780
2019-11-29,0.6762
2019-12-02,0.6819
2019-12-03,0.6841
2019-12-04,0.6851
2019-12-05,0.6834
2019-12-06,0.6836
2019-12-09,0.6832
2019-12-10,0.6814
2019-12-11,0.6864
2019-12-12,0.6896
2019-12-13,0.6871
2019-12-16,0.6887
2019-12-17,0.6847
2019-12-18,0.6854
2019-12-19,0.6887
2019-12-20,0.6902
2019-12-23,0.6917
2019-12-24,0.6923
2019-12-26,0.6939
2019-12-27,0.6978
2019-12-30,0.7004
2019-12-31,0.7030
2020-01-02,0.6982
2020-01-03,0.6963
2020-01-06,0.6932
2020-01-07,0.6861
2020-01-08,0.6870
2020-01-09,0.6856
2020-01-10,0.6903
2020-01-13,0.6911
2020-01-14,0.6905
2020-01-15,0.6911
2020-01-16,0.6895
2020-01-17,0.6877
2020-01-21,0.6853
2020-01-22,0.6836
2020-01-23,0.6834
2020-01-24,0.6824
2020-01-27,0.6757
2020-01-28,0.6753
2020-01-29,0.6744
2020-01-30,0.6715
2020-01-31,0.6699
2020-02-03,0.6688
2020-02-04,0.6732
2020-02-05,0.6748
2020-02-06,0.6738
2020-02-07,0.6679
2020-02-10,0.6674
2020-02-11,0.6712
2020-02-12,0.6746
2020-02-13,0.6729
2020-02-14,0.6719
2020-02-18,0.6693
2020-02-19,0.6673
2020-02-20,0.6618
2020-02-21,0.6632
2020-02-24,0.6612
2020-02-25,0.6597
2020-02-26,0.6556
2020-02-27,0.6582
2020-02-28,0.6495
2020-03-02,0.6524
2020-03-03,0.6617
2020-03-04,0.6609
2020-03-05,0.6600
2020-03-06,0.6634
2020-03-09,0.6654
2020-03-10,0.6470
2020-03-11,0.6512
2020-03-12,0.6280
2020-03-13,0.6161
2020-03-16,0.6132
2020-03-17,0.5976
2020-03-18,0.5820
2020-03-19,0.5859
2020-03-20,0.5841
2020-03-23,0.5755
2020-03-24,0.5927
2020-03-25,0.5957
2020-03-26,0.6054
2020-03-27,0.6126
2020-03-30,0.6145
2020-03-31,0.6139
2020-04-01,0.6107
2020-04-02,0.6047
2020-04-03,0.6001
2020-04-06,0.6096
2020-04-07,0.6187
2020-04-08,0.6223
2020-04-09,0.6309
2020-04-10,0.6349
2020-04-13,0.6380
2020-04-14,0.6419
2020-04-15,0.6328
2020-04-16,0.6301
2020-04-17,0.6357
2020-04-20,0.6387
2020-04-21,0.6279
2020-04-22,0.6320
2020-04-23,0.6388
2020-04-24,0.6377
2020-04-27,0.6460
2020-04-28,0.6499
2020-04-29,0.6536
2020-04-30,0.6523
2020-05-01,0.6424
2020-05-04,0.6412
2020-05-05,0.6458
2020-05-06,0.6414
2020-05-07,0.6500
2020-05-08,0.6543
2020-05-11,0.6472
2020-05-12,0.6509
2020-05-13,0.6463
2020-05-14,0.6422
2020-05-15,0.6410
2020-05-18,0.6513
2020-05-19,0.6567
2020-05-20,0.6601
2020-05-21,0.6562
2020-05-22,0.6528
2020-05-26,0.6644
2020-05-27,0.6590
2020-05-28,0.6664
2020-05-29,0.6642
2020-06-01,0.6785
2020-06-02,0.6872
2020-06-03,0.6948
2020-06-04,0.6944
2020-06-05,0.6983
2020-06-08,0.6995
2020-06-09,0.6964
2020-06-10,0.7004
2020-06-11,0.6889
2020-06-12,0.6854
2020-06-15,0.6851
2020-06-16,0.6886
2020-06-17,0.6897
2020-06-18,0.6847
2020-06-19,0.6856
2020-06-22,0.6904
2020-06-23,0.6955
2020-06-24,0.6875
2020-06-25,0.6868
2020-06-26,0.6861
2020-06-29,0.6851
2020-06-30,0.6893
2020-07-01,0.6917
2020-07-02,0.6921
2020-07-06,0.6973
2020-07-07,0.6972
2020-07-08,0.6967
2020-07-09,0.6958
2020-07-10,0.6959
2020-07-13,0.6983
2020-07-14,0.6970
2020-07-15,0.7003
2020-07-16,0.7008
2020-07-17,0.6994
2020-07-20,0.7009
2020-07-21,0.7111
2020-07-22,0.7147
2020-07-23,0.7136
2020-07-24,0.7094
2020-07-27,0.7134
2020-07-28,0.7161
2020-07-29,0.7168
2020-07-30,0.7139
2020-07-31,0.7159
2020-08-03,0.7107
2020-08-04,0.7150
2020-08-05,0.7216
2020-08-06,0.7228
2020-08-07,0.7163
2020-08-10,0.7162
2020-08-11,0.7155
2020-08-12,0.7167
2020-08-13,0.7160
2020-08-14,0.7168
2020-08-17,0.7211
2020-08-18,0.7235
2020-08-19,0.7234
2020-08-20,0.7178
2020-08-21,0.7156
2020-08-24,0.7174
2020-08-25,0.7180
2020-08-26,0.7222
2020-08-27,0.7260
2020-08-28,0.7352
2020-08-31,0.7388
2020-09-01,0.7375
2020-09-02,0.7319
2020-09-03,0.7272
2020-09-04,0.7264
2020-09-08,0.7230
2020-09-09,0.7275
2020-09-10,0.7287
2020-09-11,0.7277
2020-09-14,0.7299
2020-09-15,0.7296
2020-09-16,0.7336
2020-09-17,0.7289
2020-09-18,0.7311
2020-09-21,0.7212
2020-09-22,0.7173
2020-09-23,0.7093
2020-09-24,0.7046
2020-09-25,0.7012
2020-09-28,0.7067
2020-09-29,0.7109
2020-09-30,0.7160
2020-10-01,0.7189
2020-10-02,0.7147
2020-10-05,0.7181
2020-10-06,0.7149
2020-10-07,0.7140
2020-10-08,0.7158
2020-10-09,0.7237
2020-10-13,0.7165
2020-10-14,0.7169
2020-10-15,0.7079
2020-10-16,0.7083
2020-10-19,0.7090
2020-10-20,0.7057
2020-10-21,0.7120
2020-10-22,0.7109
2020-10-23,0.7118
2020-10-26,0.7120
2020-10-27,0.7143
2020-10-28,0.7067
2020-10-29,0.7006
2020-10-30,0.7023
2020-11-02,0.7045
2020-11-03,0.7159
2020-11-04,0.7192
2020-11-05,0.7265
2020-11-06,0.7268
2020-11-09,0.7275
2020-11-10,0.7273
2020-11-12,0.7267
2020-11-13,0.7256
2020-11-16,0.7323
2020-11-17,0.7298
2020-11-18,0.7325
2020-11-19,0.7265
2020-11-20,0.7315
2020-11-23,0.7279
2020-11-24,0.7351
2020-11-25,0.7360
2020-11-30,0.7351
2020-12-01,0.7355
2020-12-02,0.7395
2020-12-03,0.7444
2020-12-04,0.7438
2020-12-07,0.7449
2020-12-08,0.7414
2020-12-09,0.7451
2020-12-10,0.7520
2020-12-11,0.7532
2020-12-14,0.7537
2020-12-15,0.7555
2020-12-16,0.7565
2020-12-17,0.7620
2020-12-18,0.7615
2020-12-21,0.7575
2020-12-22,0.7531
2020-12-23,0.7587
2020-12-28,0.7568
2020-12-29,0.7623
2020-12-30,0.7682
2020-12-31,0.7709
2021-01-04,0.7657
2021-01-05,0.7752
2021-01-06,0.7787
2021-01-07,0.7755
2021-01-08,0.7778
2021-01-11,0.7714
2021-01-12,0.7732
2021-01-13,0.7740
2021-01-14,0.7790
2021-01-15,0.7723
2021-01-19,0.7703
2021-01-21,0.7754
2021-01-22,0.7728
2021-01-25,0.7703
2021-01-26,0.7737
2021-01-27,0.7680
2021-01-28,0.7674
2021-01-29,0.7661
2021-02-01,0.7625
2021-02-02,0.7586
2021-02-03,0.7622
2021-02-04,0.7599
2021-02-05,0.7656
2021-02-08,0.7702
2021-02-09,0.7724
2021-02-10,0.7735
2021-02-11,0.7756
2021-02-12,0.7762
2021-02-16,0.7759
2021-02-17,0.7740
2021-02-18,0.7750
2021-02-19,0.7873
2021-02-22,0.7916
2021-02-23,0.7902
2021-02-24,0.7935
2021-02-25,0.7953
2021-02-26,0.7720
2021-03-01,0.7776
2021-03-02,0.7822
2021-03-03,0.7808
2021-03-04,0.7807
2021-03-05,0.7665
2021-03-08,0.7653
2021-03-09,0.7699
2021-03-10,0.7721
2021-03-11,0.7773
2021-03-12,0.7772
2021-03-15,0.7729
2021-03-16,0.7748
2021-03-17,0.7709
2021-03-18,0.7795
2021-03-19,0.7746
2021-03-22,0.7750
2021-03-23,0.7682
2021-03-24,0.7622
2021-03-25,0.7570
2021-03-26,0.7629
2021-03-29,0.7643
2021-03-30,0.7607
2021-03-31,0.7613
2021-04-01,0.7604
2021-04-02,0.7607
2021-04-05,0.7657
2021-04-06,0.7660
2021-04-07,0.7627
2021-04-08,0.7640
2021-04-09,0.7631
2021-04-12,0.7620
2021-04-13,0.7629
2021-04-14,0.7731
2021-04-15,0.7746
2021-04-16,0.7734
2021-04-19,0.7759
2021-04-20,0.7742
2021-04-21,0.7755
2021-04-22,0.7725
2021-04-23,0.7739
2021-04-26,0.7802
2021-04-27,0.7775
2021-04-28,0.7768
2021-04-29,0.7759
2021-04-30,0.7716
2021-05-03,0.7758
2021-05-04,0.7688
2021-05-05,0.7750
2021-05-06,0.7768
2021-05-07,0.7861
2021-05-10,0.7876
2021-05-11,0.7843
2021-05-12,0.7734
2021-05-13,0.7723
2021-05-14,0.7776
2021-05-17,0.7765
2021-05-18,0.7782
2021-05-19,0.7753
2021-05-20,0.7780
2021-05-21,0.7731
2021-05-24,0.7741
2021-05-25,0.7741
2021-05-26,0.7743
2021-05-27,0.7736
2021-05-28,0.7706
2021-06-01,0.7765
2021-06-02,0.7752
2021-06-03,0.7652
2021-06-04,0.7742
2021-06-07,0.7765
2021-06-08,0.7741
2021-06-09,0.7731
2021-06-10,0.7752
2021-06-11,0.7696
2021-06-14,0.7718
2021-06-15,0.7676
2021-06-16,0.7712
2021-06-17,0.7555
2021-06-18,0.7499
2021-06-21,0.7528
2021-06-22,0.7528
2021-06-23,0.7589
2021-06-24,0.7573
2021-06-25,0.7595
2021-06-28,0.7574
2021-06-29,0.7517
2021-06-30,0.7496
2021-07-01,0.7472
2021-07-02,0.7499
2021-07-06,0.7496
2021-07-07,0.7485
2021-07-08,0.7437
2021-07-09,0.7480
2021-07-12,0.7490
2021-07-13,0.7471
2021-07-14,0.7469
2021-07-15,0.7436
2021-07-16,0.7409
2021-07-19,0.7343
2021-07-20,0.7325
2021-07-21,0.7357
2021-07-22,0.7365
2021-07-23,0.7364
2021-07-26,0.7386
2021-07-27,0.7368
2021-07-28,0.7349
2021-07-29,0.7400
2021-07-30,0.7351
2021-08-02,0.7364
2021-08-03,0.7385
2021-08-04,0.7378
2021-08-05,0.7409
2021-08-06,0.7357
2021-08-09,0.7340
2021-08-10,0.7346
2021-08-11,0.7374
2021-08-12,0.7345
2021-08-13,0.7377
2021-08-16,0.7339
2021-08-17,0.7254
2021-08-18,0.7236
2021-08-19,0.7170
2021-08-20,0.7133
2021-08-23,0.7208
2021-08-24,0.7254
2021-08-25,0.7266
2021-08-26,0.7248
2021-08-27,0.7309
2021-08-30,0.7295
2021-08-31,0.7307
2021-09-01,0.7369
2021-09-02,0.7401
2021-09-03,0.7442
2021-09-07,0.7394
2021-09-08,0.7368
2021-09-09,0.7380
2021-09-10,0.7366
2021-09-13,0.7370
2021-09-14,0.7335
2021-09-15,0.7324
2021-09-16,0.7288
2021-09-17,0.7276
2021-09-20,0.7248
2021-09-21,0.7240
2021-09-22,0.7264
2021-09-23,0.7297
2021-09-24,0.7258
2021-09-27,0.7284
2021-09-28,0.7237
2021-09-29,0.7190
2021-09-30,0.7228
2021-10-01,0.7271
2021-10-04,0.7284
2021-10-05,0.7295
2021-10-06,0.7258
2021-10-07,0.7321
2021-10-08,0.7308
2021-10-12,0.7366
2021-10-13,0.7371
2021-10-14,0.7419
2021-10-15,0.7416
2021-10-18,0.7417
2021-10-19,0.7481
2021-10-20,0.7512
2021-10-21,0.7492
2021-10-22,0.7463
2021-10-25,0.7494
2021-10-26,0.7497
2021-10-27,0.7515
2021-10-28,0.7547
2021-10-29,0.7510
2021-11-01,0.7534
2021-11-02,0.7429
2021-11-03,0.7430
2021-11-04,0.7402
2021-11-05,0.7408
2021-11-08,0.7429
2021-11-09,0.7369
2021-11-10,0.7360
2021-11-12,0.7322
2021-11-15,0.7361
2021-11-16,0.7309
2021-11-17,0.7273
2021-11-18,0.7270
2021-11-19,0.7257
2021-11-22,0.7235
2021-11-23,0.7222
2021-11-24,0.7197
2021-11-26,0.7124
2021-11-29,0.7119
2021-11-30,0.7088
2021-12-01,0.7132
2021-12-02,0.7097
2021-12-03,0.7011
2021-12-06,0.7044
2021-12-07,0.7113
2021-12-08,0.7167
2021-12-09,0.7146
2021-12-10,0.7177
2021-12-13,0.7123
2021-12-14,0.7103
2021-12-15,0.7125
2021-12-16,0.7191
2021-12-17,0.7148
2021-12-20,0.7117
2021-12-21,0.7140
2021-12-22,0.7215
2021-12-23,0.7240
2021-12-27,0.7238
2021-12-28,0.7233
2021-12-29,0.7241
2021-12-30,0.7260
2022-01-03,0.7195
2022-01-04,0.7241
2022-01-05,0.7260
2022-01-06,0.7168
2022-01-07,0.7186
2022-01-10,0.7170
2022-01-11,0.7198
2022-01-12,0.7277
2022-01-13,0.7299
2022-01-14,0.7222
2022-01-18,0.7188
2022-01-19,0.7226
2022-01-20,0.7266
2022-01-21,0.7199
2022-01-24,0.7101
2022-01-25,0.7138
2022-01-26,0.7167
2022-01-27,0.7044
2022-01-28,0.6998
2022-01-31,0.7064
2022-02-01,0.7110
2022-02-02,0.7125
2022-02-03,0.7143
2022-02-04,0.7067
2022-02-07,0.7111
2022-02-08,0.7133
2022-02-09,0.7183
2022-02-10,0.7245
2022-02-11,0.7177
2022-02-14,0.7127
2022-02-15,0.7152
2022-02-16,0.7183
2022-02-17,0.7195
2022-02-18,0.7174
2022-02-22,0.7226
2022-02-23,0.7245
2022-02-24,0.7133
2022-02-25,0.7218
2022-02-28,0.7261
2022-03-01,0.7248
2022-03-02,0.7282
2022-03-03,0.7315
2022-03-04,0.7362
2022-03-07,0.7321
2022-03-08,0.7262
2022-03-09,0.7323
2022-03-10,0.7361
2022-03-11,0.7308
2022-03-14,0.7228
2022-03-15,0.7196
2022-03-16,0.7251
2022-03-17,0.7388
2022-03-18,0.7402
2022-03-21,0.7419
2022-03-22,0.7438
2022-03-23,0.7498
2022-03-24,0.7505
2022-03-25,0.7501
2022-03-28,0.7483
2022-03-29,0.7502
2022-03-30,0.7508
2022-03-31,0.7499
2022-04-01,0.7493
2022-04-04,0.7549
2022-04-05,0.7598
2022-04-06,0.7542
2022-04-07,0.7472
2022-04-08,0.7461
2022-04-11,0.7420
2022-04-12,0.7480
2022-04-13,0.7430
2022-04-14,0.7404
2022-04-15,0.7401
2022-04-18,0.7353
2022-04-19,0.7370
2022-04-20,0.7452
2022-04-21,0.7376
2022-04-22,0.7257
2022-04-25,0.7147
2022-04-26,0.7145
2022-04-27,0.7124
2022-04-28,0.7077
2022-04-29,0.7101
2022-05-02,0.7058
2022-05-03,0.7099
2022-05-04,0.7145
2022-05-05,0.7089
2022-05-06,0.7091
2022-05-09,0.6989
2022-05-10,0.6925
2022-05-11,0.6994
2022-05-12,0.6852
2022-05-13,0.6918
2022-05-16,0.6943
2022-05-17,0.7013
2022-05-18,0.6998
2022-05-19,0.7057
2022-05-20,0.7032
2022-05-23,0.7099
2022-05-24,0.7081
2022-05-25,0.7066
2022-05-26,0.7078
2022-05-27,0.7155
2022-05-31,0.7188
2022-06-01,0.7164
2022-06-02,0.7251
2022-06-03,0.7209
2022-06-06,0.7207
2022-06-07,0.7212
2022-06-08,0.7218
2022-06-09,0.7120
2022-06-10,0.7054
2022-06-13,0.6946
2022-06-14,0.6885
2022-06-15,0.6946
2022-06-16,0.6998
2022-06-17,0.6923
2022-06-21,0.6975
2022-06-22,0.6934
2022-06-23,0.6893
2022-06-24,0.6946
2022-06-27,0.6942
2022-06-28,0.6916
2022-06-29,0.6887
2022-06-30,0.6905
2022-07-01,0.6789
2022-07-05,0.6774
2022-07-06,0.6774
2022-07-07,0.6827
2022-07-08,0.6858
2022-07-11,0.6750
2022-07-12,0.6772
2022-07-13,0.6786
2022-07-14,0.6744
2022-07-15,0.6793
2022-07-18,0.6838
2022-07-19,0.6904
2022-07-20,0.6897
2022-07-21,0.6896
2022-07-22,0.6948
2022-07-25,0.6954
2022-07-26,0.6939
2022-07-27,0.6921
2022-07-28,0.6971
2022-07-29,0.6985
2022-08-01,0.7035
2022-08-02,0.6952
2022-08-03,0.6930
2022-08-04,0.6966
2022-08-05,0.6901
2022-08-08,0.6991
2022-08-09,0.6973
2022-08-10,0.7099
2022-08-11,0.7119
2022-08-12,0.7106
2022-08-15,0.7046
2022-08-16,0.7011
2022-08-17,0.6914
2022-08-18,0.6930
2022-08-19,0.6870
2022-08-22,0.6871
2022-08-23,0.6943
2022-08-24,0.6913
2022-08-25,0.6961
2022-08-26,0.6921
2022-08-29,0.6909
2022-08-30,0.6875
2022-08-31,0.6866
2022-09-01,0.6780
2022-09-02,0.6845
2022-09-06,0.6750
2022-09-07,0.6731
2022-09-08,0.6740
2022-09-09,0.6846
2022-09-12,0.6887
2022-09-13,0.6763
2022-09-14,0.6752
2022-09-15,0.6708
2022-09-16,0.6703
2022-09-19,0.6694
2022-09-20,0.6704
2022-09-21,0.6661
2022-09-22,0.6636
2022-09-23,0.6543
2022-09-26,0.6466
2022-09-27,0.6445
2022-09-28,0.6512
2022-09-29,0.6490
2022-09-30,0.6437
2022-10-03,0.6501
2022-10-04,0.6493
2022-10-05,0.6466
2022-10-06,0.6411
2022-10-07,0.6393
2022-10-11,0.6329
2022-10-12,0.6273
2022-10-13,0.6285
2022-10-14,0.6219
2022-10-17,0.6309
2022-10-18,0.6281
2022-10-19,0.6271
2022-10-20,0.6336
2022-10-21,0.6368
2022-10-24,0.6310
2022-10-25,0.6395
2022-10-26,0.6499
2022-10-27,0.6477
2022-10-28,0.6396
2022-10-31,0.6394
2022-11-01,0.6392
2022-11-02,0.6412
2022-11-03,0.6294
2022-11-04,0.6435
2022-11-07,0.6463
2022-11-08,0.6545
2022-11-09,0.6448
2022-11-10,0.6590
2022-11-14,0.6695
2022-11-15,0.6779
2022-11-16,0.6733
2022-11-17,0.6666
2022-11-18,0.6681
2022-11-21,0.6590
2022-11-22,0.6635
2022-11-23,0.6707
2022-11-25,0.6750
2022-11-28,0.6694
2022-11-29,0.6691
2022-11-30,0.6717
2022-12-01,0.6804
2022-12-02,0.6801
2022-12-05,0.6713
2022-12-06,0.6698
2022-12-07,0.6722
2022-12-08,0.6769
2022-12-09,0.6809
2022-12-12,0.6741
2022-12-13,0.6869
2022-12-14,0.6861
2022-12-15,0.6700
2022-12-16,0.6691
2022-12-19,0.6711
2022-12-20,0.6665
2022-12-21,0.6696
2022-12-22,0.6671
2022-12-23,0.6710
2022-12-27,0.6742
2022-12-28,0.6752
2022-12-29,0.6779
2022-12-30,0.6805
2023-01-03,0.6730
2023-01-04,0.6860
2023-01-05,0.6744
2023-01-06,0.6864
2023-01-09,0.6948
2023-01-10,0.6883
2023-01-11,0.6899
2023-01-12,0.6946
2023-01-13,0.6950
2023-01-17,0.6986
2023-01-18,0.6988
2023-01-19,0.6887
2023-01-20,0.6953
2023-01-23,0.7024
2023-01-24,0.7042
2023-01-25,0.7083
2023-01-26,0.7094
2023-01-27,0.7102
2023-01-30,0.7076
2023-01-31,0.7050
2023-02-01,0.7067
2023-02-02,0.7084
2023-02-03,0.6946
2023-02-06,0.6869
2023-02-07,0.6925
2023-02-08,0.6941
2023-02-09,0.6964
2023-02-10,0.6919
2023-02-13,0.6964
2023-02-14,0.6962
2023-02-15,0.6893
2023-02-16,0.6889
2023-02-17,0.6865
2023-02-21,0.6865
2023-02-22,0.6819
2023-02-23,0.6787
2023-02-24,0.6735
2023-02-27,0.6730
2023-02-28,0.6744
2023-03-01,0.6768
2023-03-02,0.6720
2023-03-03,0.6759
2023-03-06,0.6738
2023-03-07,0.6600
2023-03-08,0.6602
2023-03-09,0.6611
2023-03-10,0.6610
2023-03-13,0.6686
2023-03-14,0.6662
2023-03-15,0.6611
2023-03-16,0.6652
2023-03-17,0.6683
2023-03-20,0.6711
2023-03-21,0.6652
2023-03-22,0.6686
2023-03-23,0.6719
2023-03-24,0.6643
2023-03-27,0.6642
2023-03-28,0.6709
2023-03-29,0.6680
2023-03-30,0.6701
2023-03-31,0.6704
2023-04-03,0.6784
2023-04-04,0.6742
2023-04-05,0.6728
2023-04-06,0.6674
2023-04-07,0.6668
2023-04-10,0.6625
2023-04-11,0.6649
2023-04-12,0.6688
2023-04-13,0.6779
2023-04-14,0.6702
2023-04-17,0.6692
2023-04-18,0.6719
2023-04-19,0.6717
2023-04-20,0.6754
2023-04-21,0.6691
2023-04-24,0.6681
2023-04-25,0.6628
2023-04-26,0.6616
2023-04-27,0.6614
2023-04-28,0.6617
2023-05-01,0.6636
2023-05-02,0.6670
2023-05-03,0.6675
2023-05-04,0.6676
2023-05-05,0.6747
2023-05-08,0.6792
2023-05-09,0.6752
2023-05-10,0.6764
2023-05-11,0.6694
2023-05-12,0.6649
2023-05-15,0.6698
2023-05-16,0.6663
2023-05-17,0.6651
2023-05-18,0.6618
2023-05-19,0.6661
2023-05-22,0.6644
2023-05-23,0.6622
2023-05-24,0.6541
2023-05-25,0.6506
2023-05-26,0.6511
2023-05-30,0.6513
2023-05-31,0.6473
2023-06-01,0.6577
2023-06-02,0.6616
2023-06-05,0.6626
2023-06-06,0.6674
2023-06-07,0.6661
2023-06-08,0.6707
2023-06-09,0.6740
2023-06-12,0.6747
2023-06-13,0.6776
2023-06-14,0.6832
2023-06-15,0.6871
2023-06-16,0.6865
2023-06-20,0.6770
2023-06-21,0.6781
2023-06-22,0.6753
2023-06-23,0.6687
2023-06-26,0.6683
2023-06-27,0.6683
2023-06-28,0.6620
2023-06-29,0.6633
2023-06-30,0.6663
2023-07-03,0.6678
2023-07-05,0.6663
2023-07-06,0.6619
2023-07-07,0.6686
2023-07-10,0.6669
2023-07-11,0.6664
2023-07-12,0.6782
2023-07-13,0.6873
2023-07-14,0.6843
2023-07-17,0.6821
2023-07-18,0.6817
2023-07-19,0.6763
2023-07-20,0.6778
2023-07-21,0.6737
2023-07-24,0.6750
2023-07-25,0.6791
2023-07-26,0.6756
2023-07-27,0.6740
2023-07-28,0.6670
2023-07-31,0.6730
2023-08-01,0.6611
2023-08-02,0.6542
2023-08-03,0.6554
2023-08-04,0.6605
2023-08-07,0.6564
2023-08-08,0.6514
2023-08-09,0.6525
2023-08-10,0.6557
2023-08-11,0.6493
2023-08-14,0.6496
2023-08-15,0.6484
2023-08-16,0.6440
2023-08-17,0.6417
2023-08-18,0.6413
2023-08-21,0.6393
2023-08-22,0.6425
2023-08-23,0.6469
2023-08-24,0.6426
2023-08-25,0.6401
2023-08-28,0.6426
2023-08-29,0.6459
2023-08-30,0.6495
2023-08-31,0.6468
2023-09-01,0.6449
2023-09-05,0.6382
2023-09-06,0.6382
2023-09-07,0.6380
2023-09-08,0.6386
2023-09-11,0.6435
2023-09-12,0.6423
2023-09-13,0.6425
2023-09-14,0.6445
2023-09-15,0.6442
2023-09-18,0.6442
2023-09-19,0.6458
2023-09-20,0.6499
2023-09-21,0.6422
2023-09-22,0.6453
2023-09-25,0.6420
2023-09-26,0.6405
2023-09-27,0.6368
2023-09-28,0.6427
2023-09-29,0.6451
2023-10-02,0.6375
2023-10-03,0.6293
2023-10-04,0.6337
2023-10-05,0.6350
2023-10-06,0.6394
2023-10-10,0.6430
2023-10-11,0.6407
2023-10-12,0.6334
2023-10-13,0.6290
2023-10-16,0.6339
2023-10-17,0.6377
2023-10-18,0.6336
2023-10-19,0.6325
2023-10-20,0.6319
2023-10-23,0.6337
2023-10-24,0.6349
2023-10-25,0.6334
2023-10-26,0.6312
2023-10-27,0.6352
2023-10-30,0.6374
2023-10-31,0.6332
2023-11-01,0.6363
2023-11-02,0.6422
2023-11-03,0.6507
2023-11-06,0.6503
2023-11-07,0.6424
2023-11-08,0.6421
2023-11-09,0.6420
2023-11-13,0.6387
2023-11-14,0.6491
2023-11-15,0.6518
2023-11-16,0.6464
2023-11-17,0.6494
2023-11-20,0.6557
2023-11-21,0.6568
2023-11-22,0.6535
2023-11-24,0.6584
2023-11-27,0.6597
2023-11-28,0.6666
2023-11-29,0.6618
2023-11-30,0.6614
2023-12-01,0.6660
2023-12-04,0.6620
2023-12-05,0.6551
2023-12-06,0.6569
2023-12-07,0.6601
2023-12-08,0.6569
2023-12-11,0.6563
2023-12-12,0.6561
2023-12-13,0.6583
2023-12-14,0.6713
2023-12-15,0.6715
2023-12-18,0.6697
2023-12-19,0.6765
2023-12-20,0.6764
2023-12-21,0.6790
2023-12-22,0.6803
2023-12-26,0.6816
2023-12-27,0.6844
2023-12-28,0.6844
2023-12-29,0.6828
2024-01-02,0.6771
2024-01-03,0.6720
2024-01-04,0.6710
2024-01-05,0.6722
2024-01-08,0.6719
2024-01-09,0.6692
2024-01-10,0.6692
2024-01-11,0.6653
2024-01-12,0.6692
2024-01-16,0.6597
2024-01-17,0.6539
2024-01-18,0.6564
2024-01-19,0.6581
2024-01-22,0.6586
2024-01-23,0.6566
2024-01-24,0.6601
2024-01-25,0.6581
2024-01-26,0.6589
2024-01-29,0.6594
2024-01-30,0.6595
2024-01-31,0.6600
2024-02-01,0.6566
2024-02-02,0.6510
2024-02-05,0.6478
2024-02-06,0.6517
2024-02-07,0.6523
2024-02-08,0.6490
2024-02-09,0.6516
2024-02-12,0.6538
2024-02-13,0.6460
2024-02-14,0.6489
2024-02-15,0.6508
2024-02-16,0.6531
2024-02-20,0.6560
2024-02-21,0.6546
2024-02-22,0.6547
2024-02-23,0.6566
2024-02-26,0.6539
2024-02-27,0.6547
2024-02-28,0.6496
2024-02-29,0.6503
2024-03-01,0.6529
2024-03-04,0.6521
2024-03-05,0.6507
2024-03-06,0.6577
2024-03-07,0.6618
2024-03-08,0.6627
2024-03-11,0.6608
2024-03-12,0.6601
2024-03-13,0.6623
2024-03-14,0.6585
2024-03-15,0.6569
2024-03-18,0.6562
2024-03-19,0.6528
2024-03-20,0.6532
2024-03-21,0.6570
2024-03-22,0.6520
2024-03-25,0.6539
2024-03-26,0.6538
2024-03-27,0.6525
2024-03-28,0.6525
2024-03-29,0.6524
2024-04-01,0.6484
2024-04-02,0.6513
2024-04-03,0.6558
2024-04-04,0.6615
2024-04-05,0.6584
2024-04-08,0.6602
2024-04-09,0.6622
2024-04-10,0.6509
2024-04-11,0.6525
2024-04-12,0.6478
2024-04-15,0.6469
2024-04-16,0.6412
2024-04-17,0.6419
2024-04-18,0.6437
2024-04-19,0.6423
2024-04-22,0.6439
2024-04-23,0.6479
2024-04-24,0.6486
2024-04-25,0.6506
2024-04-26,0.6528
2024-04-29,0.6565
2024-04-30,0.6490
2024-05-01,0.6496
2024-05-02,0.6547
2024-05-03,0.6616
2024-05-06,0.6623
2024-05-07,0.6608
2024-05-08,0.6585
2024-05-09,0.6606
2024-05-10,0.6607
2024-05-13,0.6611
2024-05-14,0.6623
2024-05-15,0.6674
2024-05-16,0.6683
2024-05-17,0.6686
2024-05-20,0.6675
2024-05-21,0.6666
2024-05-22,0.6636
2024-05-23,0.6629
2024-05-24,0.6629
2024-05-28,0.6666
2024-05-29,0.6619
2024-05-30,0.6642
2024-05-31,0.6640
2024-06-03,0.6683
2024-06-04,0.6643
2024-06-05,0.6643
2024-06-06,0.6662
2024-06-07,0.6589
2024-06-10,0.6611
2024-06-11,0.6598
2024-06-12,0.6701
2024-06-13,0.6641
2024-06-14,0.6617
2024-06-17,0.6599
2024-06-18,0.6652
2024-06-20,0.6656
2024-06-21,0.6640
2024-06-24,0.6655
2024-06-25,0.6642
2024-06-26,0.6649
2024-06-27,0.6648
2024-06-28,0.6677
2024-07-01,0.6656
2024-07-02,0.6659
2024-07-03,0.6719
2024-07-05,0.6743
2024-07-08,0.6741
2024-07-09,0.6737
2024-07-10,0.6747
2024-07-11,0.6776
2024-07-12,0.6788
2024-07-15,0.6778
2024-07-16,0.6726
2024-07-17,0.6728
2024-07-18,0.6726
2024-07-19,0.6694
2024-07-22,0.6639
2024-07-23,0.6619
2024-07-24,0.6600
2024-07-25,0.6552
2024-07-26,0.6557
2024-07-29,0.6540
2024-07-30,0.6537
2024-07-31,0.6535
2024-08-01,0.6522
2024-08-02,0.6528
2024-08-05,0.6488
2024-08-06,0.6519
2024-08-07,0.6552
2024-08-08,0.6581
2024-08-09,0.6577
2024-08-12,0.6595
2024-08-13,0.6613
2024-08-14,0.6617
2024-08-15,0.6629
2024-08-16,0.6649
2024-08-19,0.6715
2024-08-20,0.6743
2024-08-21,0.6753
2024-08-22,0.6708
2024-08-23,0.6777
2024-08-26,0.6772
2024-08-27,0.6779
2024-08-28,0.6781
2024-08-29,0.6807
2024-08-30,0.6765
2024-09-03,0.6721
2024-09-04,0.6724
2024-09-05,0.6727
2024-09-06,0.6678
2024-09-09,0.6664
2024-09-10,0.6652
2024-09-11,0.6650
2024-09-12,0.6696
2024-09-13,0.6712
2024-09-16,0.6739
2024-09-17,0.6754
2024-09-18,0.6761
2024-09-19,0.6817
2024-09-20,0.6806
2024-09-23,0.6848
2024-09-24,0.6873
2024-09-25,0.6837
2024-09-26,0.6902
2024-09-27,0.6911
2024-09-30,0.6934
2024-10-01,0.6888
2024-10-02,0.6895
2024-10-03,0.6843
2024-10-04,0.6797
2024-10-07,0.6761
2024-10-08,0.6738
2024-10-09,0.6725
2024-10-10,0.6733
2024-10-11,0.6756
2024-10-15,0.6709
2024-10-16,0.6662
2024-10-17,0.6702
2024-10-18,0.6700
2024-10-21,0.6661
2024-10-22,0.6679
2024-10-23,0.6632
2024-10-24,0.6629
2024-10-25,0.6621
2024-10-28,0.6587
2024-10-29,0.6557
2024-10-30,0.6584
2024-10-31,0.6550
2024-11-01,0.6582
2024-11-04,0.6584
2024-11-05,0.6633
2024-11-06,0.6573
2024-11-07,0.6671
2024-11-08,0.6573
2024-11-12,0.6522
2024-11-13,0.6486
2024-11-14,0.6473
2024-11-15,0.6461
2024-11-18,0.6498
2024-11-19,0.6524
2024-11-20,0.6494
2024-11-21,0.6503
2024-11-22,0.6495
2024-11-25,0.6491
2024-11-26,0.6461
2024-11-27,0.6492
2024-11-29,0.6513
2024-12-02,0.6459
2024-12-03,0.6483
2024-12-04,0.6436
2024-12-05,0.6439
2024-12-06,0.6382
2024-12-09,0.6462
2024-12-10,0.6377
2024-12-11,0.6378
2024-12-12,0.6388
2024-12-13,0.6359
2024-12-16,0.6372
2024-12-17,0.6339
2024-12-18,0.6302
2024-12-19,0.6252
2024-12-20,0.6266
2024-12-23,0.6243
2024-12-24,0.6230
2024-12-26,0.6227
2024-12-27,0.6208
2024-12-30,0.6219
2024-12-31,0.6185
2025-01-02,0.6208
2025-01-03,0.6209
2025-01-06,0.6251
2025-01-07,0.6245
2025-01-08,0.6217
2025-01-09,0.6196
2025-01-10,0.6147
2025-01-13,0.6155
2025-01-14,0.6184
2025-01-15,0.6218
2025-01-16,0.6221
2025-01-17,0.6210
2025-01-21,0.6272
2025-01-22,0.6276
2025-01-23,0.6287
2025-01-24,0.6322
2025-01-27,0.6280
2025-01-28,0.6247
2025-01-29,0.6228
2025-01-30,0.6233
2025-01-31,0.6242
2025-02-03,0.6177
2025-02-04,0.6257
2025-02-05,0.6296
2025-02-06,0.6279
2025-02-07,0.6264
2025-02-10,0.6279
2025-02-11,0.6294
2025-02-12,0.6285
2025-02-13,0.6292
2025-02-14,0.6359
2025-02-18,0.6353
2025-02-19,0.6341
2025-02-20,0.6393
2025-02-21,0.6376
2025-02-24,0.6361
2025-02-25,0.6332
2025-02-26,0.6323
2025-02-27,0.6256
2025-02-28,0.6211
2025-03-03,0.6249
2025-03-04,0.6212
2025-03-05,0.6312
2025-03-06,0.6348
2025-03-07,0.6300
2025-03-10,0.6308
2025-03-11,0.6285
2025-03-12,0.6306
2025-03-13,0.6289
2025-03-14,0.6318
2025-03-17,0.6375
2025-03-18,0.6356
2025-03-19,0.6331
2025-03-20,0.6301
2025-03-21,0.6266
2025-03-24,0.6281
2025-03-25,0.6309
2025-03-26,0.6307
2025-03-27,0.6308
2025-03-28,0.6300
2025-03-31,0.6236
2025-04-01,0.6278
2025-04-02,0.6300
2025-04-03,0.6352
2025-04-04,0.6043
2025-04-07,0.5980
2025-04-08,0.6015
2025-04-09,0.6003
2025-04-10,0.6192
2025-04-11,0.6245
2025-04-14,0.6312
2025-04-15,0.6358
2025-04-16,0.6389
2025-04-17,0.6382
2025-04-18,0.6375
2025-04-21,0.6412
2025-04-22,0.6391
2025-04-23,0.6376
2025-04-24,0.6396
2025-04-25,0.6396
2025-04-28,0.6413
2025-04-29,0.6385
2025-04-30,0.6399
2025-05-01,0.6373
2025-05-02,0.6447
2025-05-05,0.6472
2025-05-06,0.6489
2025-05-07,0.6462
2025-05-08,0.6416
2025-05-09,0.6425
2025-05-12,0.6369
2025-05-13,0.6472
2025-05-14,0.6448
2025-05-15,0.6403
2025-05-16,0.6395
2025-05-19,0.6458
2025-05-20,0.6405
2025-05-21,0.6468
2025-05-22,0.6417
2025-05-23,0.6478
2025-05-27,0.6437
2025-05-28,0.6417
2025-05-29,0.6443
2025-05-30,0.6434
2025-06-02,0.6493
2025-06-03,0.6462
2025-06-04,0.6499
2025-06-05,0.6517
2025-06-06,0.6496
2025-06-09,0.6526
2025-06-10,0.6523
2025-06-11,0.6521
2025-06-12,0.6525
2025-06-13,0.6513
2025-06-16,0.6536
2025-06-17,0.6515
2025-06-18,0.6520
2025-06-20,0.6455
2025-06-23,0.6423
2025-06-24,0.6499
2025-06-25,0.6493
2025-06-26,0.6555
2025-06-27,0.6535
2025-06-30,0.6573
2025-07-01,0.6572
2025-07-02,0.6571
2025-07-03,0.6576
2025-07-07,0.6513
2025-07-08,0.6524
2025-07-09,0.6532
2025-07-10,0.6572
2025-07-11,0.6577
2025-07-14,0.6550
2025-07-15,0.6520
2025-07-16,0.6522
2025-07-17,0.6485
2025-07-18,0.6527
2025-07-21,0.6531
2025-07-22,0.6556
2025-07-23,0.6590
2025-07-24,0.6601
2025-07-25,0.6556
2025-07-28,0.6519
2025-07-29,0.6509
2025-07-30,0.6452
2025-07-31,0.6430
2025-08-01,0.6468
2025-08-04,0.6467
2025-08-05,0.6472
2025-08-06,0.6504
2025-08-07,0.6502
2025-08-08,0.6528
2025-08-11,0.6512
2025-08-12,0.6532
2025-08-13,0.6546
2025-08-14,0.6490
2025-08-15,0.6515
2025-08-18,0.6494
2025-08-19,0.6459
2025-08-20,0.6428
2025-08-21,0.6421
2025-08-22,0.6485
2025-08-25,0.6500
2025-08-26,0.6500
2025-08-27,0.6496
2025-08-28,0.6532
2025-08-29,0.6545
2025-09-02,0.6516
2025-09-03,0.6550
2025-09-04,0.6514
2025-09-05,0.6570
2025-09-08,0.6592
2025-09-09,0.6593
2025-09-10,0.6626
2025-09-11,0.6658
2025-09-12,0.6636
2025-09-15,0.6671
2025-09-16,0.6678
2025-09-17,0.6668
2025-09-18,0.6613
2025-09-19,0.6597
2025-09-22,0.6591
2025-09-23,0.6601
2025-09-24,0.6587
2025-09-25,0.6550
2025-09-26,0.6546
2025-09-29,0.6578
2025-09-30,0.6614
2025-10-01,0.6605
2025-10-02,0.6588
2025-10-03,0.6615
2025-10-06,0.6620
2025-10-07,0.6587
2025-10-08,0.6582
2025-10-09,0.6556
2025-10-10,0.6495
2025-10-14,0.6492
2025-10-15,0.6514
2025-10-16,0.6492
2025-10-17,0.6486
2025-10-20,0.6517
2025-10-21,0.6499
2025-10-22,0.6502
2025-10-23,0.6516
2025-10-24,0.6511
2025-10-27,0.6554
2025-10-28,0.6585
2025-10-29,0.6613
2025-10-30,0.6555
2025-10-31,0.6546
2025-11-03,0.6537
2025-11-04,0.6502
2025-11-05,0.6512
2025-11-06,0.6468
2025-11-07,0.6476
2025-11-10,0.6522
2025-11-12,0.6541
2025-11-13,0.6553
2025-11-14,0.6549
2025-11-17,0.6516
2025-11-18,0.6505
2025-11-19,0.6468
2025-11-20,0.6448
2025-11-21,0.6457
2025-11-24,0.6456
2025-11-25,0.6453
2025-11-26,0.6516
2025-11-28,0.6552
2025-12-01,0.6554
2025-12-02,0.6556
2025-12-03,0.6592
2025-12-04,0.6619
2025-12-05,0.6635
2025-12-08,0.6622
2025-12-09,0.6653
2025-12-10,0.6642
2025-12-11,0.6671
2025-12-12,0.6640
2025-12-15,0.6643
2025-12-16,0.6637
2025-12-17,0.6608
2025-12-18,0.6619
2025-12-19,0.6617
2025-12-22,0.6657
2025-12-23,0.6694
2025-12-24,0.6707
2025-12-26,0.6710
2025-12-29,0.6690
2025-12-30,0.6703
2025-12-31,0.6669
2026-01-02,0.6684
2026-01-05,0.6718
2026-01-06,0.6732
2026-01-07,0.6728
2026-01-08,0.6694
2026-01-09,0.6684
2026-01-12,0.6716
2026-01-13,0.6684
2026-01-14,0.6679
2026-01-15,0.6705
2026-01-16,0.6682
2026-01-20,0.6743
2026-01-21,0.6765
2026-01-22,0.6834
2026-01-23,0.6878
2026-01-26,0.6933
2026-01-27,0.6978
2026-01-28,0.6993
2026-01-29,0.7010
2026-01-30,0.6995
2026-02-02,0.6960
2026-02-03,0.7019
2026-02-04,0.6996
2026-02-05,0.6974
2026-02-06,0.7014
2026-02-09,0.7087
2026-02-10,0.7072
2026-02-11,0.7129
2026-02-12,0.7097
2026-02-13,0.7068
2026-02-17,0.7064
2026-02-18,0.7065
2026-02-19,0.7050
2026-02-20,0.7082
2026-02-23,0.7061
2026-02-24,0.7067
2026-02-25,0.7116
2026-02-26,0.7092
2026-02-27,0.7121
2026-03-02,0.7067
2026-03-03,0.7026
2026-03-04,0.7072
2026-03-05,0.7005
2026-03-06,0.7022
2026-03-09,0.7046
2026-03-10,0.7150
2026-03-11,0.7150
2026-03-12,0.7089
2026-03-13,0.7004
2026-03-16,0.7055
2026-03-17,0.7100
2026-03-18,0.7078
2026-03-19,0.7036
2026-03-20,0.7033
2026-03-23,0.6995
2026-03-24,0.6965
2026-03-25,0.6962
2026-03-26,0.6913
2026-03-27,0.6890
2026-03-30,0.6858
2026-03-31,0.6855
2026-04-01,0.6944
2026-04-02,0.6911
2026-04-03,0.6894
2026-04-06,0.6919
2026-04-07,0.6944
2026-04-08,0.7062
2026-04-09,0.7086
2026-04-10,0.7078
2026-04-13,0.7063
2026-04-14,0.7131
2026-04-15,0.7161
2026-04-16,0.7170
2026-04-17,0.7192
2026-04-20,0.7175
2026-04-21,0.7164
2026-04-22,0.7162
2026-04-23,0.7156
2026-04-24,0.7154
2026-04-27,0.7193
2026-04-28,0.7181
2026-04-29,0.7130
2026-04-30,0.7184
2026-05-01,0.7219
2026-05-04,0.7167
2026-05-05,0.7196
2026-05-06,0.7235
2026-05-07,0.7241
2026-05-08,0.7242
2026-05-11,0.7259
2026-05-12,0.7224
2026-05-13,0.7263
2026-05-14,0.7225
2026-05-15,0.7151
2026-05-18,0.7160
2026-05-19,0.7101
2026-05-20,0.7152
2026-05-21,0.7134
2026-05-22,0.7132
2026-05-26,0.7162
2026-05-27,0.7137
2026-05-28,0.7159
2026-05-29,0.7191
2026-06-01,0.7153
2026-06-02,0.7186
2026-06-03,0.7139
2026-06-04,0.7143
2026-06-05,0.7060
2026-06-08,0.7055
2026-06-09,0.7028
2026-06-10,0.7022
2026-06-11,0.6995
2026-06-12,0.7047
2026-06-15,0.7081
2026-06-16,0.7077
2026-06-17,0.7073
2026-06-18,0.7019
2026-06-22,0.7000
2026-06-23,0.6926
2026-06-24,0.6895
2026-06-25,0.6922
2026-06-26,0.6907
2026-06-29,0.6885
2026-06-30,0.6915
2026-07-01,0.6903
2026-07-02,0.6931
2026-07-06,0.6944
2026-07-07,0.6935
2026-07-08,0.6911
2026-07-09,0.6944
2026-07-10,0.6959
2026-07-13,0.6935
2026-07-14,0.6974
2026-07-15,0.6999
2026-07-16,0.7004
2026-07-17,0.6985
2026-07-20,0.7004
2026-07-21,0.7008
2026-07-22,0.7000
2026-07-23,0.6975
2026-07-24,0.6996
2026-07-27,0.6993
2026-07-28,0.6984
2026-07-29,0.6930
2026-07-30,0.7022
2026-07-31,0.7026
2026-08-03,0.7001
2026-08-04,0.7038
2026-08-05,0.7051
2026-08-06,0.7028
2026-08-07,0.7064
2026-08-10,0.7066
2026-08-11,0.7062
2026-08-12,0.7070
2026-08-13,0.7058
2026-08-14,0.7091
2026-08-17,0.7113
2026-08-18,0.7097
2026-08-19,0.7123
2026-08-20,0.7120
2026-08-21,0.7178
2026-08-24,0.7151
2026-08-25,0.7160
2026-08-26,0.7169
2026-08-27,0.7198
2026-08-28,0.7166
2026-08-31,0.7166
2026-09-01,0.7161
2026-09-02,0.7169
2026-09-03,0.7199
2026-09-04,0.7209
2026-09-08,0.7218
2026-09-09,0.7219
2026-09-10,0.7171
2026-09-11,0.7174
2026-09-14,0.7141
2026-09-15,0.7128
2026-09-16,0.7132
2026-09-17,0.7114
2026-09-18,0.7111
""")
print('wrote data/fed_h10_audusd.csv')

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
data_csv = "data/fed_h10_audusd.csv"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 9,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. Last episode's question

Flip 400 coins: the standard deviation of the number of heads is $\sqrt{400}/2$.

In [ ]:
out["answer"] = {"flips": 400, "sd_400": math.sqrt(400) / 2}
out["answer"]

## 3. A tree turns into a bell curve

Take Episode 7's tree with 250 steps (roughly one per trading day) and look at where the share can end up and how likely each end point is under the tree's odds. The chart groups the end points into \$5 bins and compares them with the **lognormal** curve: the shape you get when the *log* of the price is normally distributed.

In [ ]:
steps = 250
dt = T / steps
u = math.exp(vol * math.sqrt(dt)); d = 1 / u
q = (math.exp(r * dt) - d) / (u - d)
k = np.arange(steps + 1)
ends = S * u ** k * d ** (steps - k)
probs = np.array([math.comb(steps, int(j)) * q ** j * (1 - q) ** (steps - j) for j in k])
edges = np.arange(40.0, 225.0, 5.0)
mass, _ = np.histogram(ends, bins=edges, weights=probs)
centres = (edges[:-1] + edges[1:]) / 2
m_ln, s_ln = math.log(S) + (r - vol ** 2 / 2) * T, vol * math.sqrt(T)
xs = np.linspace(40, 220, 361)
lognormal = np.exp(-(np.log(xs) - m_ln) ** 2 / (2 * s_ln ** 2)) / (xs * s_ln * math.sqrt(2 * math.pi))
out["tree"] = {"steps": steps, "ends": int(len(ends)), "bar_x": centres.tolist(), "bar_y": (mass / 5).tolist(),
               "curve_x": xs.tolist(), "curve_y": lognormal.tolist(),
               "ticks": [{"x": float(x), "label": f"${x:.0f}"} for x in range(40, 221, 30)],
               "mean_end": float((ends * probs).sum()), "forward": S * math.exp(r * T),
               "binned_share": float(mass.sum())}
# One dot per end price: its probability spread over the dollars halfway to its neighbours (a density per $1).
# Tree nodes are evenly spaced in log price, not in dollars, so $ bins would catch one or two nodes each.
inside = [j for j in range(1, steps) if 40 <= ends[j] <= 220]
out["tree"]["dots_x"] = [float(ends[j]) for j in inside]
out["tree"]["dots_y"] = [float(probs[j] / ((ends[j + 1] - ends[j - 1]) / 2)) for j in inside]
assert abs(out["tree"]["mean_end"] - out["tree"]["forward"]) < 1e-9
{k_: out["tree"][k_] for k_ in ("ends", "mean_end", "forward", "binned_share")}

## 4. Volatility, defined

**Volatility** is the standard deviation of the share's log returns over a year. Over shorter periods the spread scales with the square root of time, so with about 252 trading days a year, a 20% volatility means a typical daily move of $20\% / \sqrt{252}$.

In [ ]:
out["vol"] = {"vol_pct": vol * 100, "daily_pct": vol / math.sqrt(252) * 100, "trading_days": 252,
              "sqrt_252": math.sqrt(252), "question_vol_pct": 16, "answer_16": 0.16 / math.sqrt(256) * 100}
out["vol"]

## 5. A lopsided range

One standard deviation either side, in log terms, is $S e^{\pm\sigma\sqrt{T}}$. In dollars that's further up than down, because a price can double but can't fall below zero. Over three months $\sqrt{T} = 1/2$, so the log-range halves.

In [ ]:
rows = []
for label, t in (("one year", 1.0), ("three months", 0.25)):
    lo, hi = S * math.exp(-vol * math.sqrt(t)), S * math.exp(vol * math.sqrt(t))
    rows.append({"horizon": label, "low": lo, "high": hi, "down": S - lo, "up": hi - S})
out["range"] = {"rows": rows, "y_lo": rows[0]["low"], "y_hi": rows[0]["high"], "y_down": rows[0]["down"], "y_up": rows[0]["up"],
                "q_lo": rows[1]["low"], "q_hi": rows[1]["high"]}
pd.DataFrame(rows)

## 6. Real data: AUD/USD

Daily log changes of the Australian dollar (Fed H.10), and their volatility over the last year: the standard deviation of daily log returns, times $\sqrt{252}$.

In [ ]:
fx = pd.read_csv(data_csv, parse_dates=["date"])
fx["ret"] = np.log(fx.audusd).diff()
rets = fx.dropna()
last = rets.tail(252)
daily_sd = float(last.ret.std())
edges = np.arange(-2.0, 2.05, 0.1)
counts, _ = np.histogram(last.ret * 100, bins=edges)
dens = counts / counts.sum() / 0.1
xs = np.linspace(-2, 2, 201)
out["fx"] = {"start": last.date.iloc[0].date().isoformat(), "end": last.date.iloc[-1].date().isoformat(),
             "days": int(len(last)), "last": float(fx.audusd.iloc[-1]),
             "daily_pct": daily_sd * 100, "rv_pct": daily_sd * math.sqrt(252) * 100,
             "rv_full_pct": float(rets.ret.std() * math.sqrt(252) * 100),
             "full_start": rets.date.iloc[0].date().isoformat(),
             "bar_x": ((edges[:-1] + edges[1:]) / 2).round(3).tolist(), "bar_y": dens.tolist(),
             "curve_x": xs.tolist(), "curve_y": [math.exp(-(x / (daily_sd * 100)) ** 2 / 2) / (daily_sd * 100 * math.sqrt(2 * math.pi)) for x in xs],
             "ticks": [{"x": float(x), "label": f"{x:+.1f}%" if x else "0"} for x in np.arange(-2.0, 2.01, 1.0)]}
{k_: v for k_, v in out["fx"].items() if not isinstance(v, list)}

## 7. Volatility moves

Three-month (63 trading day) realised volatility through time. Black-Scholes assumes one constant volatility; the data say it changes a lot. Part 3 deals with that.

In [ ]:
roll = (rets.set_index("date").ret.rolling(63).std() * math.sqrt(252) * 100).dropna()
decimal_year = lambda d: d.year + (d.dayofyear - 1) / 365
weekly = roll.iloc[::5]
peak_end = roll.idxmax()
out["rolling"] = {"t": [round(decimal_year(d), 4) for d in weekly.index], "vol_pct": weekly.round(3).tolist(),
                  "min_pct": float(roll.min()), "max_pct": float(roll.max()),
                  "peak_end": peak_end.date().isoformat(), "peak_start": rets.date[rets.date <= peak_end].iloc[-63].date().isoformat(),
                  "ticks": [{"x": float(y), "label": str(y)} for y in range(2018, 2027, 2)]}
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(roll.index, roll.values); ax.set(title="AUD/USD 3-month realised volatility, % (Source: Federal Reserve Board, H.10)"); ax.grid(alpha=.3)
{k_: v for k_, v in out["rolling"].items() if not isinstance(v, list)}

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")